# Deep-Dive Exploratory Data Analysis (EDA)
## Retail Customer Behavioral Analytics

This notebook performs a 'perfect' exploration of the dataset to ensure production-grade readiness.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('ggplot')

sys.path.append(os.path.abspath('../src'))
from utils import load_raw_data, validate_schema, identify_outliers

### 1. Unified Data Loading

In [ ]:
df = load_raw_data('../data/raw/RetailCustomers.csv')
validate_schema(df)

### 2. Structural Analysis
Check memory usage and data types to optimize the pipeline.

In [ ]:
print(f"Memory Usage: {df.memory_usage().sum() / 1024**2:.2f} MB")
print("\n--- Data Types ---")
print(df.dtypes.value_counts())

### 3. Missing Value & Cardinality Analysis

In [ ]:
null_counts = df.isnull().sum()
unique_counts = df.nunique()

stats_df = pd.DataFrame({
    'Missing': null_counts,
    'Unique Values': unique_counts,
    'Cardinality (%)': (unique_counts / len(df)) * 100
}).sort_values(by='Missing', ascending=False)

stats_df.head(15)

### 4. Advanced Outlier & Skewness Analysis

In [ ]:
outlier_stats = identify_outliers(df)
outlier_df = pd.DataFrame(outlier_stats).T.sort_values(by='skewness', ascending=False)
outlier_df.head(10)

### 5. Distribution of Key Features (RFM)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df['Recency'], kde=True, ax=axes[0], color='blue')
axes[0].set_title('Recency Distribution')

sns.histplot(df['Frequency'], kde=True, ax=axes[1], color='green')
axes[1].set_xlim(0, 50) # Zoom in
axes[1].set_title('Frequency Distribution (Zoomed)')

sns.histplot(df['MonetaryTotal'], kde=True, ax=axes[2], color='red')
axes[2].set_xlim(0, 10000) # Zoom in
axes[2].set_title('Monetary Distribution (Zoomed)')

plt.tight_layout()
plt.show()

### 6. Target Analysis & Correlations

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='Churn', y='MonetaryTotal', data=df)
plt.ylim(0, 5000)
plt.title('MonetaryTotal by Churn Status')
plt.show()

### 7. Multicollinearity Check
Identify features with correlation > 0.8 to prioritize removal during preprocessing.

In [ ]:
corr_matrix = df.select_dtypes(include=[np.number]).corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.85)]

print(f"Highly correlated features (>0.85): {to_drop}")